# LeetCode 2574 — Left and Right Sum Differences bằng Python

Link đề: https://leetcode.com/problems/left-and-right-sum-differences/ · Độ khó: **Easy** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def leftRightDifference(self, nums: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu `leftSum[i]` (tổng các phần tử **bên trái** vị trí `i`) và `rightSum[i]` (tổng các phần tử **bên phải**), **không** tính `nums[i]`.
2. Dùng **slicing** (`nums[:i]`, `nums[i+1:]`) và hàm `sum` để viết lời giải vét cạn, rồi thấy vì sao nó làm lại việc cộng nhiều lần.
3. Nắm kỹ thuật **prefix sum** (tổng tiền tố): tính `leftSum` một lượt, và `rightSum` bằng công thức `total - left - nums[i]`.
4. Viết cách tiết kiệm bộ nhớ: chỉ giữ **một biến `left`** chạy dọc mảng cùng `total`, và cách dùng `itertools.accumulate`.
5. Đối chiếu 4 cách bằng test ngẫu nhiên có seed.
6. Tránh các lỗi quen thuộc: tính cả `nums[i]` vào tổng, quên `abs`, dùng `nums[i:]` thay `nums[i+1:]`, cập nhật `left` sai thời điểm.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (chỉ số bắt đầu từ 0). Hãy tìm mảng `answer` thoả:

- `answer.length == nums.length`;
- `answer[i] = |leftSum[i] - rightSum[i]|`.

Trong đó:

- `leftSum[i]` là **tổng các phần tử bên trái** chỉ số `i` trong `nums`. Nếu không có phần tử nào bên trái thì `leftSum[i] = 0`.
- `rightSum[i]` là **tổng các phần tử bên phải** chỉ số `i` trong `nums`. Nếu không có phần tử nào bên phải thì `rightSum[i] = 0`.

Trả về mảng `answer`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[10,4,8,3]` | `[15,1,11,22]` | `leftSum = [0,10,14,22]`, `rightSum = [15,11,3,0]`, hiệu tuyệt đối: `15, 1, 11, 22` |
| 2 | `[1]` | `[0]` | một phần tử: bên trái và bên phải đều rỗng, `|0 - 0| = 0` |

Constraints: `1 <= nums.length <= 1000`; `1 <= nums[i] <= 10^5`.

In [1]:
nums = [10, 4, 8, 3]
print("nums =", nums)
print("leftSum  theo đề (viết tay):", [0, 10, 14, 22])
print("rightSum theo đề (viết tay):", [15, 11, 3, 0])
print("answer   mong đợi          :", [15, 1, 11, 22])

nums = [10, 4, 8, 3]
leftSum  theo đề (viết tay): [0, 10, 14, 22]
rightSum theo đề (viết tay): [15, 11, 3, 0]
answer   mong đợi          : [15, 1, 11, 22]


## 3. Kiến thức nền

### 3.1. Slicing `nums[a:b]`

`nums[a:b]` lấy các phần tử có chỉ số từ `a` tới `b - 1` (**không** gồm `b`). Bỏ `a` thì bắt đầu từ đầu, bỏ `b` thì chạy tới cuối.

- `nums[:i]` = các phần tử **trước** vị trí `i` (đúng là "bên trái").
- `nums[i + 1:]` = các phần tử **sau** vị trí `i` (đúng là "bên phải"). Còn `nums[i:]` thì **gồm cả `nums[i]`** — rất dễ nhầm.

In [2]:
nums = [10, 4, 8, 3]
i = 2
print("nums[:i]     =", nums[:i])       # [10, 4]  bên trái của vị trí 2
print("nums[i+1:]   =", nums[i + 1:])   # [3]      bên phải của vị trí 2
print("nums[i:]     =", nums[i:])       # [8, 3]   gồm cả chính nó -> KHÔNG phải bên phải
print("nums[:0]     =", nums[:0], "-> rỗng, sum =", sum(nums[:0]))

nums[:i]     = [10, 4]
nums[i+1:]   = [3]
nums[i:]     = [8, 3]
nums[:0]     = [] -> rỗng, sum = 0


### 3.2. `sum`, `abs` và tổng của list rỗng

- `sum(list)` cộng tất cả phần tử; `sum([]) = 0` nên ta **không cần** xử lý riêng trường hợp "không có phần tử bên trái/phải".
- `abs(x)` là giá trị tuyệt đối: `abs(-7) = 7`.

In [3]:
print(sum([10, 4]), sum([]), abs(-7), abs(5 - 12), abs(12 - 5))

14 0 7 7 7


### 3.3. Prefix sum (tổng tiền tố)

`prefix[i]` = tổng các phần tử đứng **trước** vị trí `i`. Công thức đệ quy: `prefix[0] = 0`, `prefix[i + 1] = prefix[i] + nums[i]`. Ta không phải cộng lại từ đầu mỗi lần mà **tận dụng kết quả cũ**.

In [4]:
nums = [10, 4, 8, 3]
prefix = [0]                         # prefix[0] = 0: trước vị trí 0 không có gì
for x in nums:
    prefix.append(prefix[-1] + x)
    print(f"thêm {x} -> prefix = {prefix}")
# prefix[i] là tổng của nums[:i]; prefix[len(nums)] là tổng toàn mảng
print("leftSum  =", prefix[:-1])
print("tổng cả mảng =", prefix[-1])

thêm 10 -> prefix = [0, 10]
thêm 4 -> prefix = [0, 10, 14]
thêm 8 -> prefix = [0, 10, 14, 22]
thêm 3 -> prefix = [0, 10, 14, 22, 25]
leftSum  = [0, 10, 14, 22]
tổng cả mảng = 25


### 3.3b. Suy ra `rightSum` từ tổng toàn mảng

Tổng cả mảng `total` chia làm ba phần: **trái + chính nó + phải**. Vậy:

`rightSum[i] = total - leftSum[i] - nums[i]`.

Ví dụ `i = 1` với `[10, 4, 8, 3]`: `total = 25`, `left = 10`, `nums[1] = 4` → `right = 25 - 10 - 4 = 11` (đúng: `8 + 3`).

In [5]:
nums = [10, 4, 8, 3]
total = sum(nums)
left = 0
for i, x in enumerate(nums):
    right = total - left - x
    print(f"i={i}: left={left}, x={x}, right={right}, |left-right|={abs(left - right)}")
    left += x          # cập nhật SAU khi dùng, để left luôn là tổng bên trái của vị trí kế tiếp

i=0: left=0, x=10, right=15, |left-right|=15
i=1: left=10, x=4, right=11, |left-right|=1
i=2: left=14, x=8, right=3, |left-right|=11
i=3: left=22, x=3, right=0, |left-right|=22


### 3.4. `itertools.accumulate`

`accumulate(nums)` trả về các tổng tích luỹ `nums[0]`, `nums[0]+nums[1]`, ... Nếu thêm `initial=0` thì dãy bắt đầu bằng `0` — chính là `prefix` ở trên.

In [6]:
from itertools import accumulate
nums = [10, 4, 8, 3]
print(list(accumulate(nums)))               # [10, 14, 22, 25]
print(list(accumulate(nums, initial=0)))    # [0, 10, 14, 22, 25]

[10, 14, 22, 25]
[0, 10, 14, 22, 25]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đứng ở từng vị trí `i`, nhìn sang **trái** cộng lại, nhìn sang **phải** cộng lại, lấy hiệu tuyệt đối. Tính tay `nums = [10, 4, 8, 3]`:

```
i=0: trái = (rỗng)=0        phải = 4+8+3 = 15   -> |0-15|  = 15
i=1: trái = 10              phải = 8+3   = 11   -> |10-11| = 1
i=2: trái = 10+4 = 14       phải = 3     = 3    -> |14-3|  = 11
i=3: trái = 10+4+8 = 22     phải = (rỗng)=0     -> |22-0|  = 22
```

In [7]:
nums = [10, 4, 8, 3]
for i in range(len(nums)):
    left_part = nums[:i]
    right_part = nums[i + 1:]
    print(f"i={i}: trái={left_part} (sum={sum(left_part)}), phải={right_part} (sum={sum(right_part)}), "
          f"đáp án={abs(sum(left_part) - sum(right_part))}")

i=0: trái=[] (sum=0), phải=[4, 8, 3] (sum=15), đáp án=15
i=1: trái=[10] (sum=10), phải=[8, 3] (sum=11), đáp án=1
i=2: trái=[10, 4] (sum=14), phải=[3] (sum=3), đáp án=11
i=3: trái=[10, 4, 8] (sum=22), phải=[] (sum=0), đáp án=22


## 5. Cách 1 — Brute Force / baseline

Làm đúng như trực giác: với mỗi `i`, cắt lát bên trái, bên phải và `sum` từng lát.

In [8]:
from typing import List

def diff_bruteforce(nums: List[int]) -> List[int]:
    answer = []
    for i in range(len(nums)):
        left = sum(nums[:i])
        right = sum(nums[i + 1:])
        answer.append(abs(left - right))
    return answer

In [9]:
print(diff_bruteforce([10, 4, 8, 3]))   # [15, 1, 11, 22]
print(diff_bruteforce([1]))             # [0]

[15, 1, 11, 22]
[0]


**Độ phức tạp:** mỗi `i` cộng khoảng `n - 1` phần tử (tổng độ dài hai lát là `n - 1`) → Time `O(n^2)`; Space `O(n)` cho lát cắt và `answer`.

**Nút thắt:** `sum(nums[:i])` và `sum(nums[:i+1])` **làm lại gần như toàn bộ phép cộng** — thay vì lấy tổng cũ rồi cộng thêm một phần tử. Với `n <= 1000` vẫn đủ nhanh (khoảng một triệu phép cộng) nhưng lãng phí.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát biến `O(n^2)` thành `O(n)`:

1. **`leftSum` tăng dần**: `leftSum[i + 1] = leftSum[i] + nums[i]`. Không cần cộng lại từ đầu.
2. **`rightSum` suy ra từ `total`**: `rightSum[i] = total - leftSum[i] - nums[i]`, với `total = sum(nums)` tính **một lần**.

Quy tắc "nếu… thì…" khi duyệt trái → phải: tại vị trí `i`, **trước khi** cộng `nums[i]` vào `left`, ta dùng `left` làm `leftSum[i]`, tính `right = total - left - nums[i]`, ghi `abs(left - right)`, **rồi mới** `left += nums[i]`.

In [10]:
nums = [10, 4, 8, 3]
total = sum(nums)
print("total =", total)
left_sums = [0, 10, 14, 22]
right_sums = [total - l - x for l, x in zip(left_sums, nums)]
print("rightSum =", right_sums)                       # [15, 11, 3, 0]
print("answer   =", [abs(l - r) for l, r in zip(left_sums, right_sums)])

total = 25
rightSum = [15, 11, 3, 0]
answer   = [15, 1, 11, 22]


## 7. Các cách tiếp theo

Ta có thêm 3 cách: **(2)** hai mảng prefix `left[]` và `right[]` rõ ràng, **(3)** chỉ một biến `left` chạy cùng `total`, **(4)** `accumulate` gọn một dòng.

### Cách 2 — Hai mảng tổng tiền tố

Giải thích từng đoạn:

1. `left[i]` được dựng từ trái sang phải: `left[i] = left[i - 1] + nums[i - 1]`.
2. `right[i]` được dựng từ phải sang trái: `right[i] = right[i + 1] + nums[i + 1]` (đi ngược để mỗi `right[i]` dùng được `right[i + 1]` đã có).
3. `answer[i] = abs(left[i] - right[i])`.

Cách này dễ kiểm tra nhất (hai mảng đúng bằng đề) nhưng tốn thêm hai mảng phụ.

In [11]:
def diff_two_arrays(nums):
    n = len(nums)
    left = [0] * n
    right = [0] * n
    for i in range(1, n):
        left[i] = left[i - 1] + nums[i - 1]
    for i in range(n - 2, -1, -1):
        right[i] = right[i + 1] + nums[i + 1]
    return [abs(left[i] - right[i]) for i in range(n)]

print(diff_two_arrays([10, 4, 8, 3]))

[15, 1, 11, 22]


In [12]:
from typing import List

class SolutionTwoArrays:
    def leftRightDifference(self, nums: List[int]) -> List[int]:
        n = len(nums)
        left = [0] * n
        right = [0] * n
        for i in range(1, n):
            left[i] = left[i - 1] + nums[i - 1]
        for i in range(n - 2, -1, -1):
            right[i] = right[i + 1] + nums[i + 1]
        return [abs(left[i] - right[i]) for i in range(n)]

Độ phức tạp Cách 2: ba lượt `O(n)` → Time `O(n)`; Space `O(n)` cho hai mảng phụ (ngoài `answer`).

### Cách 3 — Một biến `left` + `total` (tối ưu bộ nhớ)

Giải thích từng đoạn:

1. `total = sum(nums)` tính một lần.
2. `left = 0` ban đầu (chưa có phần tử nào bên trái vị trí 0).
3. Mỗi `x` ở vị trí `i`: `right = total - left - x`, ghi `abs(left - right)`, rồi `left += x`.

Không cần mảng phụ nào; `answer` chính là kết quả phải trả về.

In [13]:
def diff_running(nums):
    total = sum(nums)
    left = 0
    answer = []
    for x in nums:
        right = total - left - x
        answer.append(abs(left - right))
        left += x
    return answer

print(diff_running([10, 4, 8, 3]))

[15, 1, 11, 22]


In [14]:
from typing import List

class SolutionRunning:
    def leftRightDifference(self, nums: List[int]) -> List[int]:
        total = sum(nums)
        left = 0
        answer = []
        for x in nums:
            right = total - left - x
            answer.append(abs(left - right))
            left += x
        return answer

Độ phức tạp Cách 3: Time `O(n)` (một lượt `sum` + một lượt duyệt); Space `O(1)` phụ (ngoài `answer`).

### Cách 4 — `itertools.accumulate`

`accumulate(nums, initial=0)` cho dãy `[0, n0, n0+n1, ...]`; bỏ phần tử cuối ta có `leftSum`. Phần tử `i` của `leftSum` là `p[i]`, `rightSum[i] = total - p[i + 1]` (vì `p[i + 1] = left + nums[i]`).

In [15]:
from itertools import accumulate

def diff_accumulate(nums):
    p = list(accumulate(nums, initial=0))     # p[i] = tổng nums[:i]
    total = p[-1]
    return [abs(p[i] - (total - p[i + 1])) for i in range(len(nums))]

print(diff_accumulate([10, 4, 8, 3]))

[15, 1, 11, 22]


In [16]:
from typing import List
from itertools import accumulate

class SolutionAccumulate:
    def leftRightDifference(self, nums: List[int]) -> List[int]:
        p = list(accumulate(nums, initial=0))
        total = p[-1]
        return [abs(p[i] - (total - p[i + 1])) for i in range(len(nums))]

Độ phức tạp Cách 4: Time `O(n)`; Space `O(n)` cho mảng `p`.

## 8. TRACE từng bước

Trace Cách 3 với `nums = [10, 4, 8, 3]`, `total = 25`.

| `i` | `x` | `left` (trước) | `right = 25 - left - x` | `abs(left - right)` | `left` (sau) |
|---|---|---|---|---|---|
| 0 | 10 | 0 | 15 | 15 | 10 |
| 1 | 4 | 10 | 11 | 1 | 14 |
| 2 | 8 | 14 | 3 | 11 | 22 |
| 3 | 3 | 22 | 0 | 22 | 25 |

Cột "`left` (trước)" chính là `leftSum[i]`; cột "`right`" chính là `rightSum[i]`. Cuối cùng `left = 25 = total`.

In [17]:
def trace_running(nums):
    total = sum(nums)
    left = 0
    answer = []
    print("total =", total)
    for i, x in enumerate(nums):
        right = total - left - x
        d = abs(left - right)
        print(f"i={i} x={x} left={left} right={right} |left-right|={d}")
        answer.append(d)
        left += x
        print(f"      -> left cập nhật thành {left}, answer={answer}")
    return answer

trace_running([10, 4, 8, 3])

total = 25
i=0 x=10 left=0 right=15 |left-right|=15
      -> left cập nhật thành 10, answer=[15]
i=1 x=4 left=10 right=11 |left-right|=1
      -> left cập nhật thành 14, answer=[15, 1]
i=2 x=8 left=14 right=3 |left-right|=11
      -> left cập nhật thành 22, answer=[15, 1, 11]
i=3 x=3 left=22 right=0 |left-right|=22
      -> left cập nhật thành 25, answer=[15, 1, 11, 22]


[15, 1, 11, 22]

In [18]:
trace_running([1])      # một phần tử: left = 0 và right = 0

total = 1
i=0 x=1 left=0 right=0 |left-right|=0
      -> left cập nhật thành 1, answer=[0]


[0]

## 9. Code nộp LeetCode

Nộp Cách 3 (một biến `left`): ngắn, `O(n)` thời gian và `O(1)` bộ nhớ phụ.

In [19]:
from typing import List

class Solution:
    def leftRightDifference(self, nums: List[int]) -> List[int]:
        total = sum(nums)
        left = 0
        answer = []
        for x in nums:
            right = total - left - x
            answer.append(abs(left - right))
            left += x
        return answer

print(Solution().leftRightDifference([10, 4, 8, 3]))
print(Solution().leftRightDifference([1]))

[15, 1, 11, 22]
[0]


## 10. Vì sao thuật toán đúng

- **Invariant:** ngay trước khi xử lý vị trí `i`, biến `left = nums[0] + ... + nums[i-1]` đúng bằng `leftSum[i]`. Ban đầu `i = 0`: `left = 0` (tổng rỗng). Sau khi xử lý `i`, ta cộng `x = nums[i]` nên `left` thành tổng của `nums[0..i]` = `leftSum[i + 1]` → invariant giữ cho vị trí kế tiếp.
- **Công thức `right`:** mọi phần tử của mảng nằm đúng một trong ba nhóm: bên trái `i`, chính `i`, bên phải `i`. Do đó `total = leftSum[i] + nums[i] + rightSum[i]`, suy ra `rightSum[i] = total - left - x`. Đây là đẳng thức, không phải ước lượng.
- **Biên:** `i = 0` ⇒ `left = 0` đúng quy ước "không có phần tử bên trái là 0"; `i = n - 1` ⇒ `right = total - (total - x) - x = 0` đúng quy ước bên phải rỗng.
- **`abs`:** đề lấy giá trị tuyệt đối nên không cần lo `left > right` hay ngược lại.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space (phụ) |
|---|---|---|
| 1. Vét cạn (cắt lát + `sum`) | `O(n^2)` | `O(n)` (lát cắt) |
| 2. Hai mảng prefix/suffix | `O(n)` | `O(n)` |
| 3. Một biến `left` + `total` | `O(n)` | `O(1)` |
| 4. `accumulate` | `O(n)` | `O(n)` |

`answer` luôn tốn `O(n)` vì là kết quả trả về; "Space phụ" không tính nó. Cách 3 là gọn và tối ưu nhất.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (một phần tử, hai phần tử, mọi phần tử bằng nhau, giá trị lớn nhất `10^5`, mảng tăng/giảm).

In [20]:
impls = [
    ("brute", diff_bruteforce),
    ("two_arrays", diff_two_arrays),
    ("running", diff_running),
    ("accumulate", diff_accumulate),
    ("class_two_arrays", SolutionTwoArrays().leftRightDifference),
    ("class_running", SolutionRunning().leftRightDifference),
    ("class_accumulate", SolutionAccumulate().leftRightDifference),
    ("submit", Solution().leftRightDifference),
]
tests = [
    ([10, 4, 8, 3], [15, 1, 11, 22]),
    ([1], [0]),
    ([5, 5], [5, 5]),                     # trái/phải: (0,5) và (5,0)
    ([1, 1, 1], [2, 0, 2]),               # vị trí giữa cân bằng -> 0
    ([100000] * 3, [200000, 0, 200000]),  # giá trị lớn nhất
    ([1, 2, 3, 4, 5], [14, 11, 6, 1, 10]),
    ([7, 1], [1, 7]),
]
for name, f in impls:
    for nums, expected in tests:
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)
    print(name, "PASS", len(tests), "test")

brute PASS 7 test
two_arrays PASS 7 test
running PASS 7 test
accumulate PASS 7 test
class_two_arrays PASS 7 test
class_running PASS 7 test
class_accumulate PASS 7 test
submit PASS 7 test


Test 2: ngẫu nhiên có seed, **đúng constraints** (`n` từ 1 đến 1000, giá trị `1..10^5`). Oracle độc lập: tính trực tiếp theo định nghĩa bằng hai vòng `for` thô.

In [21]:
import random
random.seed(2574)

def oracle(nums):
    res = []
    for i in range(len(nums)):
        l = 0
        for k in range(0, i):
            l += nums[k]
        r = 0
        for k in range(i + 1, len(nums)):
            r += nums[k]
        res.append(l - r if l >= r else r - l)
    return res

for _ in range(200):
    n = random.choice([1, 2, 3, 10, 50, 300])
    nums = [random.randint(1, 100000) for _ in range(n)]
    expected = oracle(nums)
    for name, f in impls:
        assert f(list(nums)) == expected, (name, nums)
print("200 test ngẫu nhiên: tất cả cách đều khớp oracle")

200 test ngẫu nhiên: tất cả cách đều khớp oracle


Test 3: tính chất — độ dài bằng `n`; mọi phần tử `>= 0`; và với `nums` có `n >= 2`, `answer[0] = total - nums[0]`, `answer[n-1] = total - nums[n-1]` (vì một bên rỗng); hàm không sửa input.

In [22]:
random.seed(11)
for _ in range(300):
    n = random.randint(2, 40)
    nums = [random.randint(1, 100000) for _ in range(n)]
    keep = list(nums)
    ans = Solution().leftRightDifference(nums)
    total = sum(keep)
    assert len(ans) == n and all(a >= 0 for a in ans)
    assert ans[0] == total - keep[0] and ans[-1] == total - keep[-1]
    assert nums == keep
print("tính chất đầu/cuối, độ dài, không sửa input: PASS")

tính chất đầu/cuối, độ dài, không sửa input: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `nums[i:]` thay cho `nums[i + 1:]` (tính cả chính nó vào bên phải)

In [23]:
def wrong_slice(nums):
    return [abs(sum(nums[:i]) - sum(nums[i:])) for i in range(len(nums))]   # nums[i:] gồm cả nums[i]

print(wrong_slice([10, 4, 8, 3]))      # [25, 5, 3, 19] không phải [15, 1, 11, 22]

[25, 5, 3, 19]


Vì sao sai: `nums[i:]` bắt đầu **từ** `i`, nên tổng bên phải có thêm `nums[i]`. Đề muốn các phần tử **sau** `i`: `nums[i + 1:]`.

In [24]:
def right_slice(nums):
    return [abs(sum(nums[:i]) - sum(nums[i + 1:])) for i in range(len(nums))]

assert right_slice([10, 4, 8, 3]) == [15, 1, 11, 22]
print("đã sửa: PASS")

đã sửa: PASS


### Lỗi 2: cập nhật `left` **trước** khi tính `right` (lệch một vị trí)

In [25]:
def wrong_update_order(nums):
    total = sum(nums)
    left = 0
    answer = []
    for x in nums:
        left += x                    # SAI: left đã gồm cả x
        right = total - left - x
        answer.append(abs(left - right))
    return answer

print(wrong_update_order([10, 4, 8, 3]))   # [5, 7, 27, 28], không phải [15, 1, 11, 22]

[5, 7, 27, 28]


Vì sao sai: sau `left += x`, `left` là tổng của `nums[0..i]` (gồm cả `x`), không còn là tổng **bên trái** `i`. Thứ tự đúng: tính `right` và ghi `answer` trước, **rồi mới** `left += x`.

### Lỗi 3: quên `abs`

In [26]:
def wrong_no_abs(nums):
    total = sum(nums)
    left = 0
    answer = []
    for x in nums:
        right = total - left - x
        answer.append(left - right)    # thiếu abs: ra số âm
        left += x
    return answer

print(wrong_no_abs([10, 4, 8, 3]))     # [-15, -1, 11, 22]

[-15, -1, 11, 22]


Vì sao sai: đề yêu cầu **giá trị tuyệt đối** `|leftSum - rightSum|`. Khi `left < right` hiệu là số âm (`-15`, `-1`). Thêm `abs(...)`.

### Lỗi 4: dựng `right` bằng `total - prefix` nhưng quên trừ `nums[i]`

In [27]:
def wrong_right_formula(nums):
    total = sum(nums)
    left = 0
    answer = []
    for x in nums:
        right = total - left           # thiếu "- x": right bị tính cả x
        answer.append(abs(left - right))
        left += x
    return answer

print(wrong_right_formula([10, 4, 8, 3]))   # [25, 5, 3, 19]

[25, 5, 3, 19]


Vì sao sai: `total - left` là tổng của phần "từ `i` trở đi" (gồm cả `nums[i]`). Phải trừ thêm `x` để chỉ còn phần **sau** `i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có cụm "**tổng bên trái / bên phải** của vị trí `i`" → **prefix sum**; hầu như luôn có công thức `right = total - left - nums[i]`.
- Muốn tránh sai lệch một vị trí, nhớ hình: `[ trái | nums[i] | phải ]` — ba khúc cộng lại bằng `total`.
- Muốn không dùng mảng phụ: duyệt một lần, giữ `left` chạy; cập nhật `left` **sau cùng** của mỗi vòng.
- Bài "pivot index" (724), "find the middle index" (1991), "product of array except self" (238) đều là cùng họ trái/phải này.
- Khi `n` nhỏ (`<= 1000`) cách vét cạn `O(n^2)` vẫn qua; nhưng hãy luyện viết `O(n)` để dùng cho bài `n = 10^5`.

## 15. Biến thể của bài

1. Thay hiệu tuyệt đối bằng **hiệu có dấu** `leftSum - rightSum` hoặc bằng **tích** `leftSum * rightSum`.
2. Tìm **chỉ số** `i` làm `|leftSum - rightSum|` **nhỏ nhất** (bài tìm "điểm cân bằng").
3. Tính tổng bên trái/bên phải trong **cửa sổ** `k` phần tử (dùng prefix sum + trừ).
4. Mảng có thể chứa **số âm** hoặc `0` (công thức vẫn đúng, chỉ cần `abs`).
5. Mảng **vòng tròn**: bên phải của phần tử cuối là phần tử đầu.
6. Nhiều truy vấn: cho `Q` cặp `(l, r)` hỏi tổng đoạn → dựng prefix một lần, trả lời `O(1)` mỗi truy vấn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space (phụ) | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | cắt lát + `sum` cho từng `i` | `O(n^2)` | `O(n)` | hiểu đề / làm oracle |
| 2. Hai mảng prefix | dựng `left[]` và `right[]` từ hai phía | `O(n)` | `O(n)` | muốn thấy rõ đúng định nghĩa đề |
| 3. Biến `left` + `total` | `right = total - left - x` | `O(n)` | `O(1)` | cách nộp gọn nhất |
| 4. `accumulate` | `p[i]`, `total - p[i+1]` | `O(n)` | `O(n)` | viết ngắn, quen thư viện |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().leftRightDifference([3, 1, 2])` bằng gì?

**Bài 2 (tính tay):** với `nums = [2, 7, 1, 5]`, lập bảng `i, left, right, answer` như mục 8.

**Bài 3 (điền TODO):** hoàn thành `left_sums(nums)` trả về mảng `leftSum` (chỉ mảng trái).

**Bài 4 (tự viết lại):** viết `balance_index(nums)` trả về **chỉ số `i` đầu tiên** có `leftSum[i] == rightSum[i]`, nếu không có trả `-1` (bài "Find Pivot Index").

**Bài 5 (mini challenge):** viết `min_diff_index(nums)` trả về chỉ số `i` làm `|leftSum[i] - rightSum[i]|` **nhỏ nhất** (nếu nhiều, lấy chỉ số nhỏ nhất).

In [28]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().leftRightDifference([3, 1, 2]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def left_sums(nums):
    result = []
    left = 0
    # TODO: với mỗi x trong nums:
    #   - thêm left vào result  (đây là leftSum của vị trí hiện tại)
    #   - rồi cộng x vào left
    return result

[3, 1, 4]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** luôn nhớ hình `[ trái | nums[i] | phải ]`. Bài 3: `left` phải được **ghi lại trước** rồi mới cộng `x`.

**Mức 2 (vừa):** Bài 4: một chỉ số cân bằng khi `left == total - left - nums[i]`. Bài 5: duyệt `answer = Solution().leftRightDifference(nums)` rồi chọn chỉ số có giá trị nhỏ nhất (`answer.index(min(answer))` trả đúng chỉ số nhỏ nhất).

**Mức 3 (gần đáp án):** Bài 1: `total = 6`; `i=0`: `0` vs `3` → `3`; `i=1`: `3` vs `2` → `1`; `i=2`: `4` vs `0` → `4` ⇒ `[3, 1, 4]`. Bài 2: `total = 15`: `i=0`: `0` vs `13` → 13; `i=1`: `2` vs `6` → 4; `i=2`: `9` vs `5` → 4; `i=3`: `10` vs `0` → 10 ⇒ `[13, 4, 4, 10]`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1: total = 6 -> [|0-3|, |3-2|, |4-0|]
assert Solution().leftRightDifference([3, 1, 2]) == [3, 1, 4]

# Bài 2: nums = [2, 7, 1, 5], total = 15
assert Solution().leftRightDifference([2, 7, 1, 5]) == [13, 4, 4, 10]
# i=0: 0 vs 13 -> 13 ; i=1: 2 vs 6 -> 4 ; i=2: 9 vs 5 -> 4 ; i=3: 10 vs 0 -> 10

# Bài 3
def left_sums_answer(nums):
    result = []
    left = 0
    for x in nums:
        result.append(left)
        left += x
    return result

assert left_sums_answer([10, 4, 8, 3]) == [0, 10, 14, 22]
assert left_sums_answer([1]) == [0]

# Bài 4
def balance_index(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        if left == total - left - x:
            return i
        left += x
    return -1

assert balance_index([1, 7, 3, 6, 5, 6]) == 3
assert balance_index([1, 2, 3]) == -1
assert balance_index([2, 1, -1]) == 0

# Bài 5
def min_diff_index(nums):
    answer = Solution().leftRightDifference(nums)
    return answer.index(min(answer))

assert min_diff_index([10, 4, 8, 3]) == 1
assert min_diff_index([1]) == 0
assert min_diff_index([1, 1, 1]) == 1
print("tất cả đáp án bài tập: PASS")

tất cả đáp án bài tập: PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "trái - chính nó - phải" (một biến chạy):**

```python
total = sum(nums)
left = 0
answer = []
for x in nums:
    right = total - left - x      # phải = tổng - trái - chính nó
    answer.append(abs(left - right))
    left += x                      # cập nhật SAU CÙNG
return answer
```

**Checklist trước khi submit:**

- [ ] `left` ban đầu bằng `0`, và được cộng `x` **sau** khi dùng?
- [ ] `right = total - left - x` (đã trừ cả `x`)?
- [ ] Có `abs(...)`?
- [ ] Nếu dùng slicing: bên phải là `nums[i + 1:]` (không phải `nums[i:]`)?
- [ ] Trả về list cùng độ dài `nums`, không sửa input?
- [ ] Đã thử `nums = [1]` (kết quả `[0]`), hai phần tử, giá trị `10^5`?
- [ ] Chữ ký đúng `leftRightDifference(self, nums: List[int]) -> List[int]`?